# Hallmark add demo

## Set up a temporary repository

Create a disposable repository and a helper that runs actual `hm` commands.
All example files stay inside the temporary directory.

In [ ]:
import os
import shlex
from pathlib import Path
from tempfile import TemporaryDirectory

from click.testing import CliRunner
from hallmark.cli import hallmark

workspace = TemporaryDirectory(prefix="hallmark-add-demo-")
root = Path(workspace.name)
runner = CliRunner()
git_env = {
    "GIT_AUTHOR_NAME": "Hallmark demo",
    "GIT_AUTHOR_EMAIL": "demo@example.invalid",
    "GIT_COMMITTER_NAME": "Hallmark demo",
    "GIT_COMMITTER_EMAIL": "demo@example.invalid",
}


def hm(*arguments, expected=0, directory=None):
    previous = Path.cwd()
    os.chdir(directory or root)
    try:
        result = runner.invoke(hallmark, list(arguments), env=git_env)
    finally:
        os.chdir(previous)
    print("$ hm " + shlex.join(arguments))
    print(result.output.rstrip())
    assert result.exit_code == expected, result.exception


hm("init")
print("Temporary repository:", root)

## Set the pattern and add selected files

Show the error when no pattern exists, then save `a{a}_i{i}.h5`.
Add one file and several files while preserving earlier staged entries.
Reject `notes.txt` because it does not match. Check status after each case. 

Before the first commit, unstaging removes the file from the staged list. Its contents remain unchanged, and status shows it as untracked. Re-add it before committing.

In [ ]:
(root / "a0_i30.h5").write_text("first file\n")

hm("add", "a0_i30.h5", expected=1)
hm("status")

hm("add", "a{a}_i{i}.h5")
hm("status")

original = (root / "a0_i30.h5").read_bytes()

hm("restore", "--staged", "a0_i30.h5")
hm("status")

assert (root / "a0_i30.h5").read_bytes() == original

hm("add", "a0_i30.h5")
hm("status")
hm("commit", "-m", "First dataset")

for name in ("a0_i60.h5", "a0_i90.h5", "a1_i30.h5", "notes.txt"):
    (root / name).write_text(f"contents of {name}\n")

hm("add", "a0_i60.h5")
hm("status")
hm("add", "a0_i90.h5", "a1_i30.h5")
hm("status")
hm("add", "notes.txt", expected=1)
hm("status")

## Add a folder

Add matching files from `results/`, including its subfolders.
Report unmatched files as skipped.
Files with the same name in different folders retain their full relative paths.

In [ ]:
(root / "results" / "nested").mkdir(parents=True)

for name in (
    "results/a0_i30.h5",
    "results/nested/a2_i60.h5",
    "results/notes.txt",
):
    (root / name).write_text(f"contents of {name}\n")

hm("add", "results/")
hm("status")
hm("commit", "-m", "Add selected files and folders")

## Add every change with `hm add .`

Stage new and deleted files from the repository root.
Repeat from a subfolder to show that additions, modifications and deletions
across the whole project are still included. Run status and commit there too.

In [ ]:
(root / "a0_i30.h5").unlink()
(root / "a2_i90.h5").write_text("new root file\n")

hm("status")
hm("add", ".")
hm("status")
hm("commit", "-m", "Add root changes")

(root / "a0_i60.h5").write_text("modified root file\n")
(root / "a1_i30.h5").unlink()
(root / "a3_i90.h5").write_text("another new root file\n")
subfolder = root / "results" / "nested"

hm("status", directory=subfolder)
hm("add", ".", directory=subfolder)
hm("status", directory=subfolder)
hm("commit", "-m", "Add changes from a subfolder", directory=subfolder)
hm("status", directory=subfolder)

## Protect the catalog and staged versions

Reject a pattern that would exclude existing entries.
Show that `--force` and `--all` are unsupported.

Edit a file after staging it. Commit must stop and request another `hm add`.
Re-add the current contents, then commit successfully.

In [ ]:
hm("add", "b{b}_i{i}.h5", expected=1)
hm("status")
hm("add", "--force", "notes.txt", expected=2)
hm("add", "--all", expected=2)
hm("status")

guard_file = root / "a4_i30.h5"
guard_file.write_text("staged version\n")
hm("add", "a4_i30.h5")
guard_file.write_text("edited after add\n")

hm("status")
hm("commit", "-m", "Reject stale staged contents", expected=1)

hm("add", "a4_i30.h5")
hm("status")
hm("commit", "-m", "Commit current contents")
hm("status")

## Unstage one file

Commit a baseline, then stage a modification, a deletion, and new files.
Unstage the modified file. Other changes stay staged.

The snapshot checks include working files and `.hm/objects/`.
Git’s internal files are excluded because its index must change.

In [ ]:
from hallmark import Repo


def working_snapshot():
    return {
        path.relative_to(root).as_posix(): path.read_bytes()
        for path in root.rglob("*")
        if path.is_file() and ".git" not in path.relative_to(root).parts
    }


calibrated = root / "calibrated"
calibrated.mkdir()

for day in (95, 96, 98):
    (calibrated / f"a8_i{day}.h5").write_text(f"original day {day}\n")

hm("add", "calibrated/")
hm("commit", "-m", "Restore demo baseline")

(calibrated / "a8_i95.h5").write_text("modified day 95\n")
(calibrated / "a8_i96.h5").unlink()
(calibrated / "a8_i97.h5").write_text("new day 97\n")
(root / "a8_i99.h5").write_text("keep this file staged\n")

hm("add", ".")
hm("status")
before = working_snapshot()

hm("restore", "--staged", "calibrated/a8_i95.h5")
hm("status")

assert working_snapshot() == before
snapshot = Repo(root).status()
assert snapshot["staged"]["modified"] == []
assert snapshot["staged"]["added"] == [
    "a8_i99.h5", "calibrated/a8_i97.h5"
]
assert snapshot["staged"]["deleted"] == ["calibrated/a8_i96.h5"]
assert "calibrated/a8_i95.h5" in snapshot["worktree"]["modified"]

## Unstage a folder or the whole project

Unstage everything under `calibrated/`. The new file becomes untracked;
the deletion becomes unstaged. The file outside that folder stays staged
and can still be committed.

Then stage the remaining changes and unstage everything with `.` from
inside a subfolder. Working files and stored objects remain unchanged.

In [ ]:
hm("restore", "--staged", "calibrated/")
hm("status")

assert working_snapshot() == before
snapshot = Repo(root).status()
assert snapshot["staged"]["added"] == ["a8_i99.h5"]
assert snapshot["staged"]["deleted"] == []
assert "calibrated/a8_i96.h5" in snapshot["worktree"]["deleted"]
assert "calibrated/a8_i97.h5" in snapshot["untracked"]

hm("commit", "-m", "Commit only the remaining staged file")
hm("status")

hm("add", ".")
hm("status")
before = working_snapshot()

hm("restore", "--staged", ".", directory=calibrated)
hm("status", directory=calibrated)

assert working_snapshot() == before
assert not any(Repo(root).status()["staged"].values())

## Reject incorrect usage

Omitting the path shows usage. Omitting `--staged` shows the supported
command. Neither error changes the staged versions or working files.

In [ ]:
hm("add", "calibrated/a8_i97.h5")
before = working_snapshot()
staged_before = {
    key: entry[:2]
    for key, entry in Repo(root).dothm.index.entries.items()
}

hm("restore", "--staged", expected=1)
hm("status")
hm("restore", ".", expected=1)
hm("status")

assert {
    key: entry[:2]
    for key, entry in Repo(root).dothm.index.entries.items()
} == staged_before
assert working_snapshot() == before

hm("restore", "--staged", ".")
hm("status")

## Clean up

Remove the temporary repository and synthetic files created by this demo.

In [ ]:
workspace.cleanup()
print("Temporary repository removed.")